In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from pyspark.sql import functions as F

In [ ]:
# =============================================================
# 1. CẤU HÌNH HỆ THỐNG (THAY ĐỔI THEO PROJECT CỦA BẠN)
# =============================================================
PROJECT_ID = "project-7f16ff1d-9026-4799-bfa"
BUCKET = "amazon-reviews-lakehouse-data"
DATASET_ID = "gold_zone"

# Danh sách các bảng nguồn trong Gold Zone
TRAIN_LABELS_TABLE = "user_product_training_data" # Bảng 943M dòng (nhãn 0, 1)
USER_DNA_TABLE = "user_dna_profiles"             # Bảng đặc trưng User
PROD_FEAT_TABLE = "product_features"             # Bảng đặc trưng Product

# Bảng đích cuối cùng
TARGET_TABLE = "master_training_features"
TEMP_GCS_BUCKET = BUCKET

# Khởi tạo Spark Session
spark = DataprocSparkSession.builder.getOrCreate()

██████████████████████████▋                                                     

In [ ]:
def build_master_training_features_refined():
    print(f" Bắt đầu quy trình Gộp Đặc trưng  vào {DATASET_ID}...")

    # ---------------------------------------------------------
    # BƯỚC 1: ĐỌC DỮ LIỆU TỪ BIGQUERY (CHỈ LẤY CỘT CẦN THIẾT)
    # ---------------------------------------------------------
    print(" Đang tải dữ liệu nguồn...")

    # Đọc bảng nhãn (Bảng gốc 1 tỷ dòng)
    df_train = spark.read.format("bigquery") \
        .load(f"{PROJECT_ID}.{DATASET_ID}.{TRAIN_LABELS_TABLE}")

    # Đọc bảng User DNA
    df_user = spark.read.format("bigquery") \
        .load(f"{PROJECT_ID}.{DATASET_ID}.{USER_DNA_TABLE}")

    # Đọc bảng Product Features
    df_prod = spark.read.format("bigquery") \
        .load(f"{PROJECT_ID}.{DATASET_ID}.{PROD_FEAT_TABLE}")

    # Chuẩn bị danh sách cột để đóng gói (Loại bỏ ID để không bị trùng)
    user_cols = [c for c in df_user.columns if c != "user_id"]
    prod_cols = [c for c in df_prod.columns if c != "parent_asin"]

    # ---------------------------------------------------------
    # BƯỚC 2: JOIN TỐI ƯU (SỬ DỤNG BROADCAST HINT)
    # ---------------------------------------------------------
    print(" Đang thực hiện Join song song (Optimized)...")

    # Join 1: Ghép User DNA vào bảng nhãn (Sử dụng Shuffle Hash Join thông thường)
    df_with_user = df_train.join(df_user, on="user_id", how="left")

    # Join 2: Ghép Product Features (Dùng BROADCAST vì bảng sản phẩm thường nhỏ hơn 8GB)
    # Kỹ thuật này giúp tránh nghẽn mạng Shuffle và tăng tốc độ cực lớn
    df_combined = df_with_user.join(F.broadcast(df_prod), on="parent_asin", how="left")

    # ---------------------------------------------------------
    # BƯỚC 3: ĐÓNG GÓI THÀNH CẤU TRÚC STRUCT (DẠNG NESTED)
    # ---------------------------------------------------------
    print(" Đang đóng gói dữ liệu thành cấu trúc [user_features] và [product_features]...")

    # F.struct() gom các cột lẻ thành 1 khối duy nhất cho mỗi đối tượng
    df_final = df_combined.select(
        F.struct("user_id", *user_cols).alias("user_features"),
        F.struct("parent_asin", *prod_cols).alias("product_features"),
        F.col("label").cast("int") # Đảm bảo label là kiểu Integer
    )

    # ---------------------------------------------------------
    # BƯỚC 4: GHI DỮ LIỆU XUỐNG BIGQUERY GOLD ZONE
    # ---------------------------------------------------------
    full_target_path = f"{PROJECT_ID}.{DATASET_ID}.{TARGET_TABLE}"
    print(f" Đang lưu bảng MASTER (943M dòng) vào BigQuery: {full_target_path}")

    # Tùy chọn 'writeMethod': 'direct' giúp ghi dữ liệu nhanh hơn nếu tài nguyên cho phép
    df_final.write.format("bigquery") \
        .option("table", full_target_path) \
        .option("temporaryGcsBucket", TEMP_GCS_BUCKET) \
        .mode("overwrite") \
        .save()

    print(f" HOÀN TẤT! Bảng '{TARGET_TABLE}' đã sẵn sàng tại BigQuery.")

    # Hiển thị kết quả kiểm tra
    df_final.printSchema()
    df_final.show(5, truncate=False)

In [ ]:
# =============================================================
# THỰC THI
# =============================================================
if __name__ == "__main__":
    build_master_training_features_refined()

 Bắt đầu quy trình Gộp Đặc trưng  vào gold_zone...
 Đang tải dữ liệu nguồn...
 Đang thực hiện Join song song (Optimized)...
 Đang đóng gói dữ liệu thành cấu trúc [user_features] và [product_features]...
 Đang lưu bảng MASTER (943M dòng) vào BigQuery: project-7f16ff1d-9026-4799-bfa.gold_zone.master_training_features


  0%|           0/1112 Tasks

 HOÀN TẤT! Bảng 'master_training_features' đã sẵn sàng tại BigQuery.
root
 |-- user_features: struct (nullable = false)
 |    |-- user_id: string (nullable = true)
 |    |-- total_reviews_count: long (nullable = true)
 |    |-- diversity_score: long (nullable = true)
 |    |-- user_activity_level: string (nullable = true)
 |    |-- avg_user_rating: double (nullable = true)
 |    |-- verified_purchase_ratio: double (nullable = true)
 |    |-- user_recency_days: double (nullable = true)
 |    |-- price_sensitivity_avg: double (nullable = true)
 |    |-- category_affinity: string (nullable = true)
 |-- product_features: struct (nullable = false)
 |    |-- parent_asin: string (nullable = true)
 |    |-- average_rating: double (nullable = true)
 |    |-- rating_number: long (nullable = true)
 |    |-- product_life_cycle_days: double (nullable = true)
 |    |-- product_velocity: long (nullable = true)
 |    |-- peak_season_month: long (nullable = true)
 |    |-- category: string (nullable = 

  0%|           0/1317 Tasks

+--------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------+-----+
|user_features                                                                                                       |product_features                                                                         |label|
+--------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------+-----+
|{AE222CWHFQPRJLM6B3MV5H3DRKZA, 2, 1, New User, 4.0, 0.5, 666.2002546296296, 17.789999961853027, Amazon Home}        |{193799452X, 4.415094339622642, 53, 3415.021435185185, 0, 1, Books}                      |0    |
|{AE222CWHFQPRJLM6B3MV5H3DRKZA, 2, 1, New User, 4.0, 0.5, 666.2002546296296, 17.789999961853027, Amazon Home}        |{B07W5NPXN6, 3.5340909